<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/projects/tutorials/01-calling-a-model.ipynb)

# Calling a model from Python

**Goal:** call a model from Python through the course's client, read what goes out and what comes back, get JSON you can trust, and handle the failures a real model brings.

This is a reference tutorial. Each section is one job: a sentence on why, short cells that each do one thing, a cell for you to edit, and prompts for your coding assistant. Nothing here is graded.

**It runs with no model and no key.** The setup cell checks for a local model. If it finds one, it prints `[live]` and the model name, and every call is real. If not, it prints `[recorded]` and a date, and replays one real run of the same model from `projects/tutorials/fixtures/01-calling-a-model.json`. The code is the same on both lanes.

Run the cells in order, from the top.

## 1. Pick a lane

The same code can talk to a fake model, a local model or a cloud model, and one setting in `.env` picks which.

**What to look at:**

- The first line the setup cell prints: `[live]` or `[recorded]`.
- Which client each lane resolves to, and that a cloud lane with no key stops before any request.
- That a key lives only in `.env`, never in a cell.

In [ ]:
# Setup: find the course, pick the lane, and print which one you are on.
import json
import os
import sys
import urllib.request
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    # Colab starts in /content with no course in it, so fetch the public copy once.
    import subprocess

    ROOT = Path("/content/dev3pack")
    if not (ROOT / "pyproject.toml").exists():
        subprocess.run(
            ["git", "clone", "-q", "--depth", "1",
             "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(ROOT)],
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
sys.path.insert(0, str(ROOT / "src"))

from bootcamp_agent.config import load_settings
from bootcamp_agent.llm import FakeLLM
from bootcamp_agent.ollama import DEFAULT_BASE_URL, DEFAULT_MODEL, OllamaClient, probe

SETTINGS = load_settings(dotenv_path=ROOT / ".env")  # reads your .env, if you have one
MODEL = DEFAULT_MODEL  # the recording was made with this model, so live uses it too
BASE_URL = os.environ.get("OLLAMA_BASE_URL") or DEFAULT_BASE_URL  # the /v1 address
NATIVE_URL = BASE_URL.removesuffix("/v1")  # Ollama's own API lives one level up
FIXTURE = ROOT / "projects" / "tutorials" / "fixtures" / "01-calling-a-model.json"
RECORDED = json.loads(FIXTURE.read_text(encoding="utf-8"))
CHECK = probe(MODEL, BASE_URL)
LIVE = CHECK.ok

if LIVE:
    print(f"[live] {MODEL}")
else:
    print(f"[recorded] {RECORDED['_provenance']['recorded']}")
    print(f"  replays one real run of {RECORDED['_provenance']['model']}. To go live: {CHECK.fix}")

In [ ]:
# Helpers: record every reply when live, replay the recording when not.
REPLIES = {}  # user prompt -> model reply, filled as you run
RAW = {}  # label -> {"request": ..., "response": ...} for calls to Ollama's own API
NOT_RECORDED = "(not in the recording: start Ollama and run live to ask something new)"


class Recorder:
    """Wraps any LLMClient and keeps each reply, keyed by the user prompt."""

    def __init__(self, inner):
        self.inner = inner

    def complete(self, system, user):
        reply = self.inner.complete(system=system, user=user)
        # Keep the first reply: a later cell that asks the same prompt replays the same one.
        REPLIES.setdefault(user, reply)
        return reply


def replay(replies):
    # Longest prompt first: a follow-up prompt can contain an earlier one, so it must match first.
    ordered = dict(sorted(replies.items(), key=lambda item: -len(item[0])))
    return FakeLLM(responses=ordered, default=NOT_RECORDED)


def ollama_post(label, path, body, live=LIVE):
    """POST to Ollama's own API when live. Replay the recorded response when not."""
    if not live:
        recorded = RECORDED["raw"][label]
        if recorded["request"] != body:
            raise LookupError(f"{label}: this request is not the recorded one. Start Ollama to send it.")
        return recorded["response"]
    request = urllib.request.Request(
        NATIVE_URL + path,
        data=json.dumps(body).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=120) as response:
        payload = json.loads(response.read().decode("utf-8"))
    RAW.setdefault(label, {"request": body, "response": payload})
    return payload


llm = Recorder(OllamaClient(model=MODEL, base_url=BASE_URL) if LIVE else replay(RECORDED["replies"]))
QUESTIONS = {
    "one_call": "What is a system message, in one sentence?",
    "try_one_call": "What is a user message, in one sentence?",
    "json": "Why must the application validate structured outputs?",
    "temperature": "Name one planet in our solar system.",
    "cap": "Explain what a token is in a language model.",
    "stop": "List three fruits, one per line.",
}
ERRORS = dict(RECORDED.get("errors", {}))  # error messages seen on the live run
print(f"llm wraps a {type(llm.inner).__name__}")

The helpers cell above is plumbing. You do not need to change it. It gives you one object, `llm`, with the same `complete(system=..., user=...)` method on both lanes. `QUESTIONS` holds every question this notebook asks, so you can see them in one place.

Now look at how the course picks a lane. `load_settings` in `src/bootcamp_agent/config.py` reads `BOOTCAMP_PROVIDER`, and `get_client` in `src/bootcamp_agent/llm.py` turns it into a client.

In [ ]:
# List the lanes the course knows, and the one your environment picked.
from bootcamp_agent.config import KNOWN_PROVIDERS

print("lanes the course knows:", KNOWN_PROVIDERS)
print("your BOOTCAMP_PROVIDER: ", SETTINGS.provider)
print("a key is set:          ", SETTINGS.api_key is not None)  # True or False, never the key

In [ ]:
# Resolve three lanes from explicit settings, with no .env involved.
from bootcamp_agent.config import ConfigError
from bootcamp_agent.llm import get_client

for provider in ("fake", "ollama", "anthropic"):
    settings = load_settings({"BOOTCAMP_PROVIDER": provider})
    try:
        client = get_client(settings)
        print(f"{provider:9} -> {type(client).__name__}, base_url={settings.base_url}")
    except ConfigError as error:
        print(f"{provider:9} -> ConfigError: {error}")

In [ ]:
# Read the names .env may set. The template ships them empty; your values go only in .env.
names = [
    line.split("=", 1)[0].lstrip("# ").strip()
    for line in (ROOT / ".env.example").read_text(encoding="utf-8").splitlines()
    if "=" in line and not line.startswith("# ") and line.strip()
]
print("settings in .env.example:", names)
print(".env is in .gitignore:", ".env" in (ROOT / ".gitignore").read_text(encoding="utf-8").split())

In [ ]:
# Try it: change "fake" to "openai" and run the cell. Read the error: it names the fix, not the key.
try:
    print(type(get_client(load_settings({"BOOTCAMP_PROVIDER": "fake"}))).__name__)
except ConfigError as error:
    print("ConfigError:", error)

> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain how BOOTCAMP_PROVIDER in .env becomes a client object, following load_settings and get_client. Do not change the code."
> - "Explain why get_client raises ConfigError before it sends any request when a key is missing. Point me to the lines."

## 2. Make one call

A chat call sends a system message (the rules) and a user message (the question), and the reply is one assistant message.

**What to look at:**

- The reply is plain text. `complete` returns a `str`.
- The request `OllamaClient` builds: which URL, and the two messages with their roles.
- In the raw response, the reply sits at `choices[0].message`, with the role `assistant`.

In [ ]:
# Send one system message and one user message, and print the reply.
SYSTEM = "You are a concise teaching assistant. Answer in one sentence."

reply = llm.complete(system=SYSTEM, user=QUESTIONS["one_call"])
print(type(reply).__name__, "|", reply)

In [ ]:
# Capture the request OllamaClient would post, without sending it anywhere.
from unittest import mock


class CannedResponse:
    """The smallest stand-in for what urlopen returns."""

    def __init__(self, body):
        self.body = body

    def read(self):
        return self.body

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        return False


SENT = []


def capture(request, timeout):
    SENT.append(request)
    answer = {"choices": [{"message": {"role": "assistant", "content": "captured"}}]}
    return CannedResponse(json.dumps(answer).encode("utf-8"))


with mock.patch("bootcamp_agent.ollama.urllib.request.urlopen", capture):
    OllamaClient().complete(system=SYSTEM, user=QUESTIONS["one_call"])

POSTED = json.loads(SENT[0].data)
print(SENT[0].get_method(), SENT[0].full_url)
print(json.dumps(POSTED, indent=2))

In [ ]:
# Send the same call by hand, and read the whole response the server returns.
raw = ollama_post("one-call", "/v1/chat/completions", POSTED | {"model": MODEL})

print("message:", json.dumps(raw["choices"][0]["message"], indent=2))
print("finish_reason:", raw["choices"][0]["finish_reason"])
print("usage:", raw["usage"])

In [ ]:
# Try it: ask the other recorded question, "try_one_call", or type your own when live.
print(llm.complete(system=SYSTEM, user=QUESTIONS["try_one_call"]))

> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain what OllamaClient.complete in src/bootcamp_agent/ollama.py posts and which field of the response it returns. Do not change the code."
> - "Explain the difference between the system message and the user message in this notebook's first call, with one example of what belongs in each."

## 3. Tune the reply

Three settings shape a reply: temperature (how random), a token cap (how long) and stop strings (where to cut), and the course client sends none of them.

**What to look at:**

- The keys the course client posts. Temperature, max tokens and stop are not among them, so the server's defaults apply.
- Temperature 0 asked twice: are the two replies the same?
- A token cap ends the reply early, and `done_reason` says `length`.
- A stop string cuts the reply at the first match, and the stop string itself is not returned.

In [ ]:
# List what the course client sends, and which tuning settings are missing.
print("keys OllamaClient posts:", sorted(POSTED))
for setting in ("temperature", "max_tokens", "stop"):
    print(f"  {setting:12} {'sent' if setting in POSTED else 'not sent: the server default applies'}")

In [ ]:
# Find the one tuning value the course adapters do set: a fixed max_tokens on the Anthropic lane.
import inspect

import bootcamp_agent.llm as llm_module

for line in inspect.getsource(llm_module).splitlines():
    if "max_tokens" in line or "temperature" in line:
        print(line.strip())

To set them yourself, send a raw request to Ollama's own chat endpoint, `/api/chat`. It takes the settings in an `options` object. Ollama calls the token cap `num_predict`. On the OpenAI-style endpoint, `/v1/chat/completions`, the same three are top-level fields named `temperature`, `max_tokens` and `stop`.

In [ ]:
# Ask the same question twice at temperature 0, and compare the two replies.
def chat(label, question, options):
    body = {
        "model": MODEL,
        "messages": [{"role": "user", "content": question}],
        "stream": False,
        "options": options,
    }
    return ollama_post(label, "/api/chat", body)


first = chat("temperature-0-a", QUESTIONS["temperature"], {"temperature": 0})
second = chat("temperature-0-b", QUESTIONS["temperature"], {"temperature": 0})
print(repr(first["message"]["content"]))
print(repr(second["message"]["content"]))
print("same reply twice:", first["message"]["content"] == second["message"]["content"])

In [ ]:
# Cap the reply at 12 tokens with num_predict, and read why it stopped.
capped = chat("cap-12", QUESTIONS["cap"], {"num_predict": 12})
print(repr(capped["message"]["content"]))
print("done_reason:", capped["done_reason"], "| tokens generated:", capped["eval_count"])

In [ ]:
# Stop at the first newline, so a list comes back as its first line only.
stopped = chat("stop-newline", QUESTIONS["stop"], {"stop": ["\n"]})
print(repr(stopped["message"]["content"]))
print("done_reason:", stopped["done_reason"])

In [ ]:
# Try it: change 12 to 40 and run it live. On the recorded lane a changed request says so.
try:
    print(repr(chat("cap-12", QUESTIONS["cap"], {"num_predict": 12})["message"]["content"]))
except LookupError as error:
    print(error)

> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain what temperature, num_predict and stop each change in an Ollama /api/chat request. Do not change the code."
> - "Explain why the course client in src/bootcamp_agent/ollama.py sends no temperature, and what that means for comparing two runs."

## 4. Ask for JSON and parse it strictly

Code cannot read a paragraph reliably, so ask for one exact JSON shape and let a strict parser reject anything else.

**What to look at:**

- The instructions the course adds to every answer prompt: one shape, and a legal way to say "I do not know".
- What `parse_research_answer` accepts, and the exact reason it gives when it rejects.
- The one tolerance: a single markdown code fence around the JSON is accepted.
- The retry: one corrective second call, then a flagged refusal. Never a loop.

In [ ]:
# Print the instructions the course adds to every answer prompt.
from bootcamp_agent.schema import ANSWER_JSON_INSTRUCTIONS, AnswerParseError, parse_research_answer

print(ANSWER_JSON_INSTRUCTIONS)

In [ ]:
# Retrieve two passages from the course corpus, ask for JSON, and keep the raw reply.
from bootcamp_agent.documents import load_corpus
from bootcamp_agent.retrieval import retrieve

DOCS = load_corpus(ROOT / "data" / "corpus")
passages = retrieve(QUESTIONS["json"], DOCS, top_k=2)
context = "\n\n".join(f"[{p.chunk.doc_id}]\n{p.chunk.text}" for p in passages)

raw_json = llm.complete(
    system="Answer using ONLY the context.\n\n" + ANSWER_JSON_INSTRUCTIONS,
    user=f"Context:\n{context}\n\nQuestion: {QUESTIONS['json']}",
)
print("retrieved:", [p.chunk.doc_id for p in passages])
print(raw_json)

In [ ]:
# Parse the reply strictly. A pass gives typed fields; anything else raises AnswerParseError.
try:
    parsed = parse_research_answer(raw_json)
    print("answer:     ", parsed.answer)
    print("citations:  ", parsed.citations)
    print("confidence: ", parsed.confidence)
    print("needs review:", parsed.needs_human_review)
except AnswerParseError as error:
    print("rejected:", error)

In [ ]:
# Feed the parser four replies a model could send, and read each verdict.
GOOD = '{"answer": "Validate at the boundary.", "citations": ["structured-outputs"], "confidence": 0.8, "needs_human_review": false}'
samples = {
    "prose": "Sure! The application should validate because models make mistakes.",
    "a markdown fence": "```json\n" + GOOD + "\n```",
    "an extra field": GOOD.replace("}", ', "source": "me"}'),
    "confidence 7": GOOD.replace("0.8", "7"),
}
for name, sample in samples.items():
    try:
        parse_research_answer(sample)
        print(f"{name:17} accepted")
    except AnswerParseError as error:
        print(f"{name:17} rejected: {error}")

In [ ]:
# Watch the course's one corrective retry, in agent.py, with a scripted model that fails once.
from bootcamp_agent.agent import answer_question

scripted = FakeLLM(
    responses={"Return ONLY the JSON object": GOOD},  # the retry prompt ends with this sentence
    default="Sure! Here is my answer in words, not JSON.",
)
result = answer_question(QUESTIONS["json"], DOCS, scripted)
for event in result.trace:
    print(f"{event.kind:9} {event.detail}")
print("model calls:", len(scripted.calls))

In [ ]:
# Try it: edit this reply (drop a field, or set needs_human_review to "no") and read the verdict.
mine = '{"answer": "Because the model is untrusted input.", "citations": [], "confidence": 0.5, "needs_human_review": true}'
try:
    print("accepted:", parse_research_answer(mine))
except AnswerParseError as error:
    print("rejected:", error)

> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain each check parse_research_answer in src/bootcamp_agent/schema.py makes, in the order it makes them. Do not change the code."
> - "Explain why answer_question in src/bootcamp_agent/agent.py retries exactly once and then refuses, instead of retrying until the JSON is valid."

## 5. Compare sentences with embeddings

An embedding turns a text into a list of numbers, so two texts with similar meaning get similar numbers, and cosine similarity scores how close they are.

**What to look at:**

- How many numbers one sentence becomes.
- Which sentence scores highest against the question, even with few words in common.
- `nomic-embed-text` wants a prefix: `search_document: ` for stored texts, `search_query: ` for questions.

In [ ]:
# Embed three sentences and one question with nomic-embed-text.
EMBED_MODEL = "nomic-embed-text"
EMBED_LIVE = LIVE and probe(EMBED_MODEL, BASE_URL).ok
SENTENCES = [
    "The loop stops when the tool budget is spent.",
    "Chunks are cut at paragraph boundaries before indexing.",
    "My cat sleeps on the keyboard every afternoon.",
]
QUERY = "When does an agent stop calling tools?"

body = {
    "model": EMBED_MODEL,
    "input": ["search_document: " + s for s in SENTENCES] + ["search_query: " + QUERY],
}
vectors = ollama_post("embed", "/api/embed", body, live=EMBED_LIVE)["embeddings"]
print("[live]" if EMBED_LIVE else "[recorded]", f"{len(vectors)} vectors of {len(vectors[0])} numbers each")
print("first five numbers of the first vector:", [round(x, 3) for x in vectors[0][:5]])

In [ ]:
# Score each sentence against the question with cosine similarity, highest first.
import math


def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b, strict=True))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))


*document_vectors, query_vector = vectors
scores = sorted(
    ((cosine(vector, query_vector), sentence) for vector, sentence in zip(document_vectors, SENTENCES, strict=True)),
    reverse=True,
)
for score, sentence in scores:
    print(f"{score:.3f}  {sentence}")

In [ ]:
# Try it: compare two sentences of your own. On the recorded lane this compares two recorded ones.
a, b = 0, 1  # indexes into SENTENCES; live, change SENTENCES above and re-run both cells
print(f"{cosine(vectors[a], vectors[b]):.3f}  {SENTENCES[a]!r} vs {SENTENCES[b]!r}")

> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain what cosine similarity measures in this notebook's embeddings cell, and why it is not a probability. Do not change the code."
> - "Explain why nomic-embed-text uses different prefixes for documents and questions. Point me to where Project 02 does the same."

## 6. Handle errors and timeouts

A real model fails in a few known ways, and each one should reach your caller as a named error or a flagged refusal, never as a traceback.

**What to look at:**

- The client raises one error type, `OllamaError`, and each message carries its fix.
- A timeout and a missing server print the same message. The client cannot tell them apart.
- The refusal at the end: no citations, confidence 0.0, flagged for review, and the cause kept aside for you.

In [ ]:
# Call a server that is not there: the client raises OllamaError, with the fix in the message.
from bootcamp_agent.ollama import OllamaError

nowhere = OllamaClient(base_url="http://127.0.0.1:9/v1", timeout=2)
try:
    nowhere.complete(system=SYSTEM, user="hello")
except OllamaError as error:
    print("OllamaError:", error)
    print("caused by:  ", type(error.__cause__).__name__)

In [ ]:
# Call a server that accepts the connection and never answers, with a 1 second deadline.
import socket
import time

silent = socket.socket()
silent.bind(("127.0.0.1", 0))
silent.listen()  # accepts connections, never replies
port = silent.getsockname()[1]

started = time.monotonic()
try:
    OllamaClient(base_url=f"http://127.0.0.1:{port}/v1", timeout=1).complete(system=SYSTEM, user="hello")
except OllamaError as error:
    print(f"after {time.monotonic() - started:.1f}s  OllamaError:", error)
    print("caused by:  ", type(error.__cause__).__name__)
finally:
    silent.close()

In [ ]:
# Ask for a model that is not pulled: the server answers HTTP 404, and the message says what to pull.
if LIVE:
    try:
        OllamaClient(model="not-a-real-model:1b", base_url=BASE_URL).complete(system=SYSTEM, user="hello")
    except OllamaError as error:
        ERRORS["wrong-model"] = str(error)
print("[live]" if LIVE else "[recorded]", "OllamaError:", ERRORS["wrong-model"])

In [ ]:
# Turn any failed call into a flagged refusal the caller can route on, keeping the cause separate.
from bootcamp_agent.schema import ResearchAnswer


def complete_or_refuse(client, system, user):
    """Return (text, None) on success, or (a flagged refusal, the cause) on failure."""
    try:
        return client.complete(system=system, user=user), None
    except (TimeoutError, OllamaError) as error:
        refusal = ResearchAnswer(
            answer="The model did not respond, so there is no answer.",
            citations=(),
            confidence=0.0,
            needs_human_review=True,
        )
        return refusal, f"{type(error).__name__}: {error}"


answer, cause = complete_or_refuse(nowhere, SYSTEM, "hello")
print("answer:", answer)
print("cause, for your logs:", cause)

In [ ]:
# Try it: point the client at a live server and a model you have, and see a success instead.
answer, cause = complete_or_refuse(nowhere, SYSTEM, QUESTIONS["one_call"])  # try: llm instead of nowhere
print(type(answer).__name__, "|", cause)

Session 2 teaches this pattern in full, and its check `ch02-e4` is yours to write. The function above is a reference with a different shape. Read [Session 2: failing closed](https://github.com/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-02-model-adapter/concepts-3.mdx) for why the refusal never names the cause in the answer text.

> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain why OllamaClient reports a timeout with the same message as a missing server, using the except clauses in src/bootcamp_agent/ollama.py. Do not change the code."
> - "Explain what a caller can do with a flagged refusal that it cannot do with a traceback. Point me to the session 2 page that covers it."

## 7. Count the cost and know where your text goes

A local model costs nothing per token and your text never leaves the machine, while a cloud model bills per token and your text goes to the provider.

**What to look at:**

- The token counts in the raw responses: `usage` on the OpenAI-style endpoint, `prompt_eval_count` and `eval_count` on Ollama's own.
- `complete` returns text only. The course client drops the counts, on every lane.
- The address the local lane talks to.

In [ ]:
# Read the token counts from responses you already have.
print("one call (OpenAI-style):", raw["usage"])
print("capped call (Ollama's own): prompt", capped["prompt_eval_count"], "| reply", capped["eval_count"])

In [ ]:
# Check where the local lane sends your text.
from urllib.parse import urlparse

host = urlparse(BASE_URL).hostname
print("the local lane posts to:", host, "| on this machine:", host in ("localhost", "127.0.0.1"))

On the cloud lanes the SDK response carries the counts too: `response.usage.input_tokens` and `output_tokens` from Anthropic, `response.usage.prompt_tokens` and `completion_tokens` from OpenAI. The course adapters in `llm.py` return text only, so to see them you read the SDK response yourself. Prices change, so read them on the provider's pricing page, never from a notebook.

In [ ]:
# Try it: put your provider's real prices here (per million tokens) to estimate one call's cost.
EXAMPLE_PRICE_IN, EXAMPLE_PRICE_OUT = 1.00, 1.00  # made-up example prices, not any provider's
tokens_in, tokens_out = raw["usage"]["prompt_tokens"], raw["usage"]["completion_tokens"]
cost = tokens_in * EXAMPLE_PRICE_IN / 1e6 + tokens_out * EXAMPLE_PRICE_OUT / 1e6
print(f"{tokens_in} in, {tokens_out} out -> {cost:.6f} at the example prices; the local lane bills 0")

> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain which fields in an Ollama response report token counts, and why OllamaClient.complete does not return them. Do not change the code."
> - "Explain what leaves my machine when BOOTCAMP_PROVIDER is anthropic, and what stays, using src/bootcamp_agent/llm.py."

## Cheat sheet

Every call in this notebook, in one table.

| Job | Call | What you get back |
|---|---|---|
| Pick a lane | `load_settings(dotenv_path=ROOT / ".env")` | `Settings(provider, model, api_key, base_url)` |
| Get a client | `get_client(settings)` | `FakeLLM`, `OllamaClient` or a cloud adapter; `ConfigError` if a key is missing |
| Check the local model | `probe(model, base_url)` | `ProbeResult`: `.ok`, and `.fix` when it is not |
| One call | `client.complete(system=..., user=...)` | the assistant's text, as a `str` |
| The raw call | POST `/v1/chat/completions` with `model`, `messages`, `stream` | `choices[0].message`, `finish_reason`, `usage` |
| Tune it | POST `/api/chat` with `options`: `temperature`, `num_predict`, `stop` | `message`, `done_reason`, `eval_count` |
| Ask for JSON | add `ANSWER_JSON_INSTRUCTIONS` to the system message | a reply that should be one JSON object |
| Parse strictly | `parse_research_answer(text)` | `ResearchAnswer`, or `AnswerParseError` with the reason |
| Answer with retry | `answer_question(question, documents, client)` | `AgentResult(answer, trace)`, at most one retry |
| Embed | POST `/api/embed` with `model` and `input` | `embeddings`: one list of numbers per input |
| Compare | `cosine(a, b)` | a score: higher means closer in meaning |
| Fail closed | `except (TimeoutError, OllamaError)` | a `ResearchAnswer` flagged for review, citing nothing |
| Count tokens | read `usage`, or `prompt_eval_count` and `eval_count` | numbers the course client does not return |

In [ ]:
# Save this run as the recording. Maintainers only: it does nothing unless TUTORIAL_RECORD=1.
from datetime import date

if LIVE and os.environ.get("TUTORIAL_RECORD") == "1":
    RECORDED = {
        "_provenance": {
            "recorded": date.today().isoformat(),
            "model": MODEL,
            "embed_model": "nomic-embed-text",
            "lane": "one real run of the local model, replayed when no model is running",
            "auth_sent": "none",
            "temperature": "not set on the course client's calls, so Ollama used its default; set to 0 only where the notebook says so",
            "keys": "replies are keyed by the full user prompt; raw calls by label, with the exact request",
            "is_evidence_of": "what this model returned for these prompts on that run, byte for byte",
            "is_not_evidence_of": "what it returns every time, what another model returns, or how fast your machine is. Run it live to see yours.",
        },
        "replies": REPLIES,
        "raw": RAW,
        "errors": ERRORS,
    }
    FIXTURE.write_text(json.dumps(RECORDED, indent=1, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"wrote {FIXTURE.relative_to(ROOT)}: {len(REPLIES)} replies, {len(RAW)} raw calls")
else:
    print("nothing saved: this cell writes only when live and TUTORIAL_RECORD=1")

## Resources

The course pages this tutorial draws on:

- [Session 2: call a model through the adapter](https://github.com/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-02-model-adapter/introduction.mdx)
- [Session 2: failing closed](https://github.com/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-02-model-adapter/concepts-3.mdx)
- [Session 3: structured outputs](https://github.com/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-03-structured-outputs/introduction.mdx)
- [The settings: config.py](https://github.com/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/src/bootcamp_agent/config.py)
- [The seam and the adapters: llm.py](https://github.com/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/src/bootcamp_agent/llm.py)
- [The local lane: ollama.py](https://github.com/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/src/bootcamp_agent/ollama.py)
- [The answer contract: schema.py](https://github.com/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/src/bootcamp_agent/schema.py)
- [The retry and the refusal: agent.py](https://github.com/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/src/bootcamp_agent/agent.py)
- [Project 02: embeddings on real filings](https://github.com/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/projects/02-sec-filings/notebook.ipynb)
- [The .env template](https://github.com/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/.env.example)
- [Ollama's API reference](https://docs.ollama.com/api/chat)

## Ask your assistant about this tutorial

> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain the difference between the [live] and [recorded] lanes in projects/tutorials/01-calling-a-model.ipynb, and what a recorded run is not evidence of. Do not change the code."
> - "Explain the path one call takes from llm.complete to the HTTP request, naming each file it passes through."
> - "Explain which tuning settings the course client sends and which it leaves to the server. Point me to the lines."
> - "Explain what parse_research_answer rejects and why a strict parser beats a lenient one. Do not write my session 3 exercise."
> - "Explain how a timeout becomes a flagged refusal, and which session teaches it. Do not write my ch02-e4 answer."